## Zero-Shot — HMM transferido a Traffic y Exchange

Usa caches HMM entrenados en ETTh1/ETTh2/Weather/Electricity para predecir en Traffic y Exchange **sin re-entrenar el HMM**. Solo se entrena el Transformer.

**Resumible**: salta experimentos ya completados.

In [ ]:
import os, sys, time, subprocess
import numpy as np

while not os.path.exists("run.py"):
    os.chdir("..")
sys.path.insert(0, ".")

PRED_LENS = [96, 336]

# Source datasets with their optimal HMM config (from K sweep)
SOURCE_CONFIGS = {
    "ETTh1": ("hmm_soft_residual", 4, "etth1"),
    "Weather": ("hmm_soft", 8, "weather"),
}

# Target zero-shot datasets
TARGETS = {
    "Traffic": {
        "root_path": "./dataset/traffic/",
        "data_path": "traffic.csv",
        "data": "custom",
        "target": "OT",
    },
    "Exchange": {
        "root_path": "./dataset/exchange_rate/",
        "data_path": "exchange_rate.csv",
        "data": "custom",
        "target": "OT",
    },
}

NON_HMM_TECHNIQUES = ["patching", "decomposition"]

TRANSFORMER_CFG = {
    "seq_len": 96,
    "label_len": 48,
    "d_model": 64,
    "n_heads": 4,
    "e_layers": 2,
    "d_ff": 128,
    "dropout": 0.1,
    "batch_size": 32,
    "learning_rate": 0.001,
    "lradj": "type1",
    "train_epochs": 10,
    "patience": 3,
}

# Count experiments
n_hmm = len(SOURCE_CONFIGS) * len(TARGETS) * len(PRED_LENS)
n_base = len(NON_HMM_TECHNIQUES) * len(TARGETS) * len(PRED_LENS)
total = n_hmm + n_base
print(f"Total experimentos: {total}")
print(f"  Zero-shot HMM: {n_hmm}")
print(f"  Baselines (patching, decomposition): {n_base}")

In [ ]:
def make_des(technique, source=None, K=None):
    if source and K:
        return f"zeroshot_{source}_{technique}_K{K}"
    return f"zeroshot_{technique}"

def result_path(target_cfg, pred_len, des):
    data = target_cfg["data"]
    cfg = TRANSFORMER_CFG
    seq = cfg["seq_len"]
    ll = cfg["label_len"]
    dm = cfg["d_model"]
    nh = cfg["n_heads"]
    el = cfg["e_layers"]
    df = cfg["d_ff"]
    setting = (
        f"plan_a_{data}_96_{pred_len}_TransformerCommon_{data}"
        f"_ftS_sl{seq}_ll{ll}_pl{pred_len}"
        f"_dm{dm}_nh{nh}_el{el}_dl1"
        f"_df{df}_expand2_dc4_fc1_ebtimeF_dtTrue_{des}_0"
    )
    return f"./results/{setting}/metrics.npy"

def run_exp(target_cfg, pred_len, technique, des, hmm_cache_path=""):
    data = target_cfg["data"]
    cfg = TRANSFORMER_CFG
    cmd = [
        "python", "-u", "run.py",
        "--task_name", "plan_a",
        "--is_training", "1",
        "--root_path", target_cfg["root_path"],
        "--data_path", target_cfg["data_path"],
        "--model_id", f"{data}_96_{pred_len}",
        "--model", "TransformerCommon",
        "--data", data,
        "--features", "S",
        "--target", target_cfg["target"],
        "--seq_len", str(cfg["seq_len"]),
        "--label_len", str(cfg["label_len"]),
        "--pred_len", str(pred_len),
        "--enc_in", "1", "--dec_in", "1", "--c_out", "1",
        "--d_model", str(cfg["d_model"]),
        "--n_heads", str(cfg["n_heads"]),
        "--e_layers", str(cfg["e_layers"]),
        "--d_ff", str(cfg["d_ff"]),
        "--dropout", str(cfg["dropout"]),
        "--batch_size", str(cfg["batch_size"]),
        "--learning_rate", str(cfg["learning_rate"]),
        "--lradj", cfg["lradj"],
        "--train_epochs", str(cfg["train_epochs"]),
        "--patience", str(cfg["patience"]),
        "--use_gpu", "0",
        "--technique", technique,
        "--des", des,
        "--itr", "1",
    ]
    if hmm_cache_path:
        cmd += ["--hmm_cache_path", hmm_cache_path]
        # hmm_k needed for embedding dimension
        k = int(hmm_cache_path.split("_K")[1].split(".")[0])
        cmd += ["--hmm_k", str(k)]

    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
    elapsed = time.time() - t0
    mse_line = [l for l in proc.stdout.split("\n") if l.startswith("mse:")]
    if mse_line:
        return elapsed, mse_line[-1]
    err = proc.stderr[-500:] if proc.stderr else proc.stdout[-500:]
    return elapsed, f"ERROR: {err}"


experiments = []

# Baselines per target
for tgt_name, tgt_cfg in TARGETS.items():
    for pl in PRED_LENS:
        for tech in NON_HMM_TECHNIQUES:
            des = make_des(tech)
            experiments.append((tgt_name, tgt_cfg, pl, tech, des, ""))

# Zero-shot HMM per source x target
for src_name, (tech, K, cache_key) in SOURCE_CONFIGS.items():
    cache_path = f"./cache/hmm_{cache_key}_K{K}.pth"
    for tgt_name, tgt_cfg in TARGETS.items():
        for pl in PRED_LENS:
            des = make_des(tech, src_name, K)
            experiments.append((tgt_name, tgt_cfg, pl, tech, des, cache_path))

total = len(experiments)
done = sum(1 for (_, tgt, pl, _, des, _) in experiments if os.path.exists(result_path(tgt, pl, des)))
print(f"Completados: {done}/{total}")

t_start = time.time()
for i, (tgt_name, tgt_cfg, pl, tech, des, cache) in enumerate(experiments, 1):
    rpath = result_path(tgt_cfg, pl, des)
    if os.path.exists(rpath):
        m = np.load(rpath)
        print(f"[{i:>2}/{total}] SKIP {tgt_name} pl={pl} {des} | MSE={m[0]:.6f}")
        continue
    print(f"[{i:>2}/{total}] RUN  {tgt_name} pl={pl} {des} ...", end=" ", flush=True)
    elapsed, result = run_exp(tgt_cfg, pl, tech, des, cache)
    print(f"{elapsed:.0f}s | {result}")

elapsed_min = (time.time() - t_start) / 60
print(f"\nTotal elapsed: {elapsed_min:.1f}min")

In [ ]:
import numpy as np, os, re
from collections import defaultdict

PRED_LENS = [96, 336]
rows = []
results_dir = "./results"
for d in os.listdir(results_dir):
    m = re.search(r"_zeroshot_([\w]+?)_0$", d)
    ds_m = re.match(r"plan_a_(\w+?)_96_(\d+)_", d)
    if not m or not ds_m:
        continue
    mpath = os.path.join(results_dir, d, "metrics.npy")
    if not os.path.exists(mpath):
        continue
    technique = m.group(1)
    dataset = ds_m.group(1)
    pred_len = int(ds_m.group(2))
    metrics = np.load(mpath)
    rows.append((dataset, technique, pred_len, float(metrics[0]), float(metrics[1])))

agg = defaultdict(lambda: defaultdict(list))
for dataset, technique, pred_len, mse, mae in rows:
    agg[dataset][technique].append((pred_len, mse, mae))

for ds in ["custom"]:
    if ds not in agg:
        print(f"Sin resultados para {ds}")
        continue
    print(f"\n=== Zero-Shot Results (data={ds}) ===")
    header = f"{'Technique':<40}"
    for pl in PRED_LENS:
        header += f" {'pl='+str(pl):>9}"
    header += f" {'AVG':>9}"
    print(header)
    print("-" * 70)
    techs = sorted(agg[ds].items(), key=lambda x: np.nanmean([r[1] for r in x[1]]))
    for technique, results in techs:
        by_pl = {pl: mse for pl, mse, mae in results}
        vals = [by_pl.get(pl, float("nan")) for pl in PRED_LENS]
        avg = np.nanmean(vals)
        row = f"{technique:<40}"
        for v in vals:
            row += f" {v:>9.6f}" if not np.isnan(v) else f" {'N/A':>9}"
        row += f" {avg:>9.6f}"
        print(row)